# Run this file on Google Colab as it will be faster

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import json
import torch
import pandas as pd
import re
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from concurrent.futures import ThreadPoolExecutor
from functools import partial

# --- 1. Extraction & Cleaning Helpers ---

def clean_html(raw_html: str) -> str:
    """Removes HTML tags and decodes HTML entities."""
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator=' ', strip=True)

def remove_emojis(text: str) -> str:
    """Removes emojis and cleans whitespace."""
    if pd.isna(text):
        return ""
    text = str(text)
    emoji_pattern = re.compile(
        "["
        "\U0001F1E0-\U0001F1FF\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF\U00002700-\U000027BF\U000024C2-\U0001F251"
        "\u200d\u2640-\u2642\u2600-\u26FF\u2300-\u23FF"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

def process_single_json(zip_path: str, filename: str) -> dict:
    """Worker function for threads: Opens zip independently to avoid handle issues."""
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            file_bytes = z.read(filename)
            json_data = json.loads(file_bytes.decode('utf-8'))

        return {
            "uuid": json_data.get("uuid"),
            "title": remove_emojis(json_data.get("title", "")),
            "description": remove_emojis(clean_html(json_data.get("description", ""))),
            "minimumYearsExperience": json_data.get("minimumYearsExperience"),
            "skills": [s.get("skill") for s in json_data.get("skills", []) if s.get("skill")],
            "employmentTypes": [e.get("employmentType") for e in json_data.get("employmentTypes", []) if e.get("employmentType")],
            "ssicCode": json_data.get("ssicCode")
        }
    except Exception:
        return None

# --- 2. Main Processing Pipeline ---

def process_zip_optimized(zip_path: str, output_file: str):
    # --- STEP 1: Identify Files ---
    print(f"Step 1: Identifying valid JSON files in {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        json_filenames = [
            f for f in z.namelist()
            if f.endswith('.json') 
            and 'data/' in f 
            and '__MACOSX' not in f 
            and not f.split('/')[-1].startswith('._')
        ]
    
    if not json_filenames:
        print("No valid JSON files found.")
        return

    # --- STEP 2: Threaded Extraction ---
    print(f"Step 2: Extracting and cleaning {len(json_filenames)} files (Threading)...")
    worker_func = partial(process_single_json, zip_path)
    
    extracted_data = []
    with ThreadPoolExecutor() as executor:
        # Using tqdm to show progress in the notebook
        results = list(tqdm(executor.map(worker_func, json_filenames), total=len(json_filenames)))
        extracted_data = [res for res in results if res is not None]

    print(f"Successfully extracted {len(extracted_data)} job records.")

    # Prepare descriptions for embeddings
    descriptions = [job["description"] for job in extracted_data]
    
    # Device detection: Priority is MPS (Mac) -> CUDA (Nvidia) -> CPU
    if torch.backends.mps.is_available():
        device = 'mps'
    elif torch.cuda.is_available():
        device = 'cuda'
    else:
        device = 'cpu'
    print(f"\nUsing device: {device.upper()}")

    # --- STEP 3a: MiniLM Embeddings ---
    print("\nStep 3a: Generating embeddings with all-MiniLM-L6-v2 (384-dim)...")
    model_minilm = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    embeddings_minilm = model_minilm.encode(
        descriptions,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    # --- STEP 3b: MPNet Embeddings ---
    print("\nStep 3b: Generating embeddings with all-mpnet-base-v2 (768-dim)...")
    model_mpnet = SentenceTransformer('all-mpnet-base-v2', device=device)
    embeddings_mpnet = model_mpnet.encode(
        descriptions,
        batch_size=64, 
        show_progress_bar=True,
        normalize_embeddings=True
    )


    print("\nStep 4: Recombining data and saving to Parquet...")
    for i, job in enumerate(extracted_data):
        job["embedding_minilm"] = embeddings_minilm[i].tolist()
        job["embedding_mpnet"] = embeddings_mpnet[i].tolist()

    df = pd.DataFrame(extracted_data)
    df.to_parquet(output_file, engine="pyarrow", index=False)

    print(f"\n Done! Processed {len(df)} jobs.")
    print(f"File saved to: {output_file}")


if __name__ == "__main__":
    # Adjust your paths as needed
    ZIP_PATH = "../data/problem2.zip"
    OUTPUT_PATH = "../data/procesimport zipfile

# Extraction & HTML Cleaning

def clean_html(raw_html: str) -> str:
    """Removes HTML tags and decodes HTML entities."""
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator=' ', strip=True)

def remove_emojis(text):
    if pd.isna(text):
        return ""
    text = str(text)

    emoji_pattern = re.compile(
        "["
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F680-\U0001F6FF"  # transport & map
        "\U0001F700-\U0001F77F"  # alchemical
        "\U0001F780-\U0001F7FF"  # geometric shapes extended
        "\U0001F800-\U0001F8FF"  # supplemental arrows-c
        "\U0001F900-\U0001F9FF"  # supplemental symbols and pictographs
        "\U0001FA00-\U0001FA6F"  # chess / symbols
        "\U0001FA70-\U0001FAFF"  # symbols and pictographs extended-a
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "\u200d"                 # zero width joiner
        "\u2640-\u2642"          # gender symbols
        "\u2600-\u26FF"          # misc symbols
        "\u2300-\u23FF"
        "]+",
        flags=re.UNICODE,
    )

    text = emoji_pattern.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def process_raw_json_bytes(file_bytes: bytes) -> dict:
    """Decodes zip bytes to JSON, extracts fields, and cleans HTML."""
    try:
        json_data = json.loads(file_bytes.decode('utf-8'))

        return {
            "uuid": json_data.get("uuid"),
            "title": remove_emojis(json_data.get("title", "")),
            "description": remove_emojis(clean_html(json_data.get("description", ""))),
            "minimumYearsExperience": json_data.get("minimumYearsExperience"),
            "skills": [s.get("skill") for s in json_data.get("skills", []) if s.get("skill")],
            "employmentTypes": [e.get("employmentType") for e in json_data.get("employmentTypes", []) if e.get("employmentType")],
            "ssicCode": json_data.get("ssicCode")
        }
    except Exception as e:
        return None


# Main Processing Pipeline
def process_zip_optimized(zip_path: str, output_file: str):
    print(f"Step 1: Reading {zip_path} directly into memory...")

    raw_file_bytes = []

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Added the fix to ignore hidden macOS system files
        json_filenames = [
            f for f in z.namelist()
            if f.endswith('.json')
            and 'data/' in f
            and '__MACOSX' not in f
            and not f.split('/')[-1].startswith('._')
        ]
        print(f"Found {len(json_filenames)} valid JSON files inside the archive.")

        if not json_filenames:
            print("No JSON files found. Exiting.")
            return

        for filename in json_filenames:
            raw_file_bytes.append(z.read(filename))

    print("Step 2: Extracting data and cleaning HTML (CPU parallelized)...")
    extracted_data = []

    with ProcessPoolExecutor() as executor:
        results = executor.map(process_raw_json_bytes, raw_file_bytes)
        for res in results:
            if res is not None:
                extracted_data.append(res)

    descriptions = [job["description"] for job in extracted_data]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nUsing device: {device.upper()}")


    # MiniLM Embeddings

    print("\nStep 3a: Generating embeddings with all-MiniLM-L6-v2 (384-dim)...")
    model_minilm = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    embeddings_minilm = model_minilm.encode(
        descriptions,
        batch_size=1024,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    del model_minilm
    if device == 'cuda':
        torch.cuda.empty_cache()


    # MPNet Embeddings
    print("\nStep 3b: Generating embeddings with all-mpnet-base-v2 (768-dim)...")
    model_mpnet = SentenceTransformer('all-mpnet-base-v2', device=device)


    embeddings_mpnet = model_mpnet.encode(
        descriptions,
        batch_size=512,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    print("\nStep 4: Recombining data and saving to Parquet...")
    for i, job in enumerate(extracted_data):
        job["embedding_minilm"] = embeddings_minilm[i].tolist()
        job["embedding_mpnet"] = embeddings_mpnet[i].tolist()

    df = pd.DataFrame(extracted_data)
    df.to_parquet(output_file, engine="pyarrow")

    print(f" Successfully processed {len(df)} jobs and saved both embeddings to {output_file}")


if __name__ == "__main__":
    ZIP_FILE = "problem2.zip"
    OUTPUT_FILE = "processed_jobs_dual_embeddings.parquet"

    process_zip_optimized(ZIP_FILE, OUTPUT_FILE)
sed_jobs_dual_embeddings.parquet"
    
    process_zip_optimized(ZIP_PATH, OUTPUT_PATH)

Step 1: Identifying valid JSON files in ../data/problem2.zip...
Step 2: Extracting and cleaning 22720 files (Threading)...


  3%|▎         | 774/22720 [02:15<1:04:12,  5.70it/s]
